## Online Shoppers Purchasing Intention Classification

**Team:**
* Brad Burton

**Course:** CISD 43 – BIG DATA (Spring, 2026)

### Problem Statement

* This project analyzes the **Online Shoppers Purchasing Intention** dataset from the UCI Machine Learning Repository. The goal is to predict whether an e-commerce website session will result in a purchase (`Revenue = True`) or not (`Revenue = False`) based on behavioral and technical session features.

* Understanding purchase intent is a critical problem in e-commerce. By accurately classifying sessions, businesses can target high-intent visitors with promotions, reduce cart abandonment, and optimize marketing spend.

* **Keywords:** Purchase intent classification, e-commerce analytics, customer behavior, big data, machine learning, MongoDB

* **Dataset Source:** UCI Machine Learning Repository — [Online Shoppers Purchasing Intention Dataset](https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset)  
  Sakar, C.O. et al. (2019). *Real-time prediction of online shoppers' purchasing intention using multilayer perceptron and LSTM recurrent neural networks.* Neural Computing and Applications.

### Required Packages

Install any missing packages by running the cell below:

```bash
pip install pandas numpy matplotlib seaborn scikit-learn pymongo
```

In [ ]:
# Uncomment and run if any packages are missing
# !pip install pandas numpy matplotlib seaborn scikit-learn pymongo

### Methodology

This project follows a standard big data analytics pipeline:

1. **Data Loading & Inspection** — Load the dataset, review shape, dtypes, and summary statistics.
2. **Exploratory Data Analysis (EDA)** — Handle missing values, encode categoricals, explore feature distributions, correlations, and class imbalance.
3. **Preprocessing** — Feature scaling, train/test split.
4. **Machine Learning Models** — Four models are trained and evaluated:
   - **Model 1:** K-Nearest Neighbors (KNN) — instance-based classification
   - **Model 2:** Logistic Regression — linear probabilistic classifier (used as the "Linear" baseline)
   - **Model 3:** Decision Tree / Random Forest — tree-based ensemble classifier
   - **Model 4:** K-Means Clustering — unsupervised grouping of visitor behavior
5. **Model Comparison** — Accuracy, precision, recall, F1-score compared across models.
6. **Conclusions** — Business insights drawn from results.

---
## Section 1 — Data Loading & Inspection

In [ ]:
# Import all required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

# Set consistent plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print("All libraries imported successfully.")

In [ ]:
# Load the dataset
# Download from: https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset
# Place the CSV file in the same directory as this notebook

df = pd.read_csv('online_shoppers_intention.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{list(df.columns)}")

In [ ]:
# Preview the first few rows
df.head()

In [ ]:
# Check data types and non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

**Conclusion (Data Inspection):** The dataset contains 12,330 rows and 18 columns — 10 numeric, 8 categorical/boolean. The target variable is `Revenue` (True/False). No major anomalies are visible in the summary statistics, though we will check for missing values and class imbalance in the EDA section.

---
## Section 2 — Exploratory Data Analysis (EDA)

In [ ]:
# --- Missing Value Analysis ---
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print("No missing values found in the dataset.")
else:
    print(missing_df)

In [ ]:
# Handle missing values if any exist
# For numeric columns: fill with median (robust to outliers)
# For categorical columns: fill with mode

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
            print(f"Filled '{col}' (numeric) with median.")
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            print(f"Filled '{col}' (categorical) with mode.")

print(f"\nMissing values remaining: {df.isnull().sum().sum()}")

In [ ]:
# --- Target Variable: Class Distribution ---
target_counts = df['Revenue'].value_counts()
target_pct = df['Revenue'].value_counts(normalize=True) * 100

print("Revenue (Purchase) Distribution:")
print(pd.DataFrame({'Count': target_counts, 'Percentage': target_pct.round(2)}))

# Plot class distribution
fig, ax = plt.subplots()
target_counts.plot(kind='bar', color=['steelblue', 'coral'], edgecolor='black', ax=ax)
ax.set_title('Target Class Distribution: Revenue (Purchase Intent)')
ax.set_xlabel('Revenue (False = No Purchase, True = Purchase)')
ax.set_ylabel('Number of Sessions')
ax.set_xticklabels(['No Purchase (False)', 'Purchase (True)'], rotation=0)
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=11)
plt.tight_layout()
plt.show()

**Conclusion (Class Distribution):** The dataset is imbalanced — approximately 84% of sessions do NOT result in a purchase, while only ~16% do. This is typical for real-world e-commerce data. This imbalance means raw accuracy can be misleading; we will pay close attention to precision, recall, and F1-score when evaluating models.

In [ ]:
# --- Distribution of Key Numeric Features ---
numeric_cols = ['Administrative', 'Informational', 'ProductRelated',
                'BounceRates', 'ExitRates', 'PageValues', 'SessionDuration']

# Check which of these exist in the dataset
numeric_cols = [c for c in numeric_cols if c in df.columns]

# Use all numeric columns if the above list doesn't match
if not numeric_cols:
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'Revenue']

df[numeric_cols].hist(bins=30, figsize=(14, 10), color='steelblue', edgecolor='black')
plt.suptitle('Distribution of Numeric Features', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Conclusion (Feature Distributions):** Most numeric features are right-skewed, meaning a large number of sessions have low values (e.g., few pages visited, low bounce rates), with a smaller number of high-engagement outlier sessions. `PageValues` in particular shows a strong right skew — most sessions have low page value scores, but the high-value sessions are likely the ones that convert to purchases.

In [ ]:
# --- PageValues by Revenue (Boxplot) ---
# PageValues is typically the strongest predictor of purchase
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=df, x='Revenue', y='PageValues', palette='muted', ax=ax)
ax.set_title('PageValues by Purchase Outcome')
ax.set_xlabel('Revenue (Purchase Made)')
ax.set_ylabel('Page Values Score')
plt.tight_layout()
plt.show()

**Conclusion (PageValues vs Revenue):** Sessions that result in a purchase have significantly higher `PageValues` scores. This confirms that `PageValues` is likely one of the most predictive features in our models — visitors who spend time on high-value product pages are far more likely to convert.

In [ ]:
# --- Bounce Rate vs Exit Rate (Scatter) ---
fig, ax = plt.subplots(figsize=(8, 5))
colors = df['Revenue'].map({True: 'coral', False: 'steelblue'})
ax.scatter(df['BounceRates'], df['ExitRates'], c=colors, alpha=0.3, s=10)
ax.set_title('Bounce Rate vs Exit Rate (colored by Purchase)')
ax.set_xlabel('Bounce Rate')
ax.set_ylabel('Exit Rate')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='coral', label='Purchase (True)'),
                   Patch(facecolor='steelblue', label='No Purchase (False)')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

**Conclusion (Bounce vs Exit Rates):** There is a strong positive correlation between bounce rate and exit rate — sessions where users leave quickly also exit at high rates. Purchasing sessions (coral) tend to cluster at low bounce and exit rates, suggesting engaged visitors are more likely to convert.

In [ ]:
# --- Visits by Month ---
month_order = ['Feb', 'Mar', 'Apr', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
month_order = [m for m in month_order if m in df['Month'].unique()]

month_revenue = df.groupby('Month')['Revenue'].value_counts(normalize=True).unstack()
month_revenue = month_revenue.reindex([m for m in month_order if m in month_revenue.index])

month_revenue.plot(kind='bar', stacked=True, color=['steelblue', 'coral'], figsize=(10, 5))
plt.title('Purchase Rate by Month')
plt.xlabel('Month')
plt.ylabel('Proportion of Sessions')
plt.legend(['No Purchase', 'Purchase'], loc='upper right')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**Conclusion (Monthly Trends):** Purchase rates vary noticeably by month. November and December show higher conversion rates, consistent with holiday shopping behavior. February also shows a modest peak, likely from Valentine's Day shopping. This seasonal pattern could be leveraged to time marketing campaigns effectively.

In [ ]:
# --- Correlation Heatmap (Numeric Features) ---
# Encode Revenue as 0/1 temporarily for correlation
df_corr = df.copy()
df_corr['Revenue'] = df_corr['Revenue'].astype(int)

numeric_df = df_corr.select_dtypes(include=np.number)
corr_matrix = numeric_df.corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

**Conclusion (Correlation Heatmap):** `PageValues` has the strongest positive correlation with `Revenue`, confirming its importance as a predictor. `BounceRates` and `ExitRates` are highly correlated with each other, suggesting potential multicollinearity. `ProductRelated` and `ProductRelated_Duration` are also highly correlated, as expected.

---
## Section 3 — Data Preprocessing

In [ ]:
# --- Encode Categorical & Boolean Features ---
df_model = df.copy()

# Encode boolean columns (Weekend, Revenue) as integers
for col in ['Weekend', 'Revenue']:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype(int)

# Encode string categorical columns with LabelEncoder
le = LabelEncoder()
for col in ['Month', 'VisitorType']:
    if col in df_model.columns:
        df_model[col] = le.fit_transform(df_model[col])
        print(f"Encoded '{col}' successfully.")

print("\nEncoding complete. Data types after encoding:")
print(df_model.dtypes)

In [ ]:
# --- Define Features (X) and Target (y) ---
X = df_model.drop(columns=['Revenue'])
y = df_model['Revenue']

print(f"Feature matrix shape: {X.shape}")
print(f"Target vector shape:  {y.shape}")
print(f"\nClass balance:\n{y.value_counts()}")

In [ ]:
# --- Train/Test Split (80/20) ---
# stratify=y ensures the class imbalance ratio is preserved in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size:  {X_train.shape[0]} samples")
print(f"Test set size:      {X_test.shape[0]} samples")

In [ ]:
# --- Feature Scaling (StandardScaler) ---
# Required for KNN and Logistic Regression; good practice for all models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)  # Use fit from training set only

print("Feature scaling complete.")
print(f"Train mean (first feature, should be ~0): {X_train_scaled[:, 0].mean():.4f}")
print(f"Train std  (first feature, should be ~1): {X_train_scaled[:, 0].std():.4f}")

**Conclusion (Preprocessing):** All categorical features were label-encoded, the dataset was split 80/20 with stratification to preserve class balance, and all features were standardized. The scaler was fit only on the training set to prevent data leakage.

---
## Section 4 — Machine Learning Models

### Model 1: K-Nearest Neighbors (KNN)

In [ ]:
# --- Find Optimal K ---
# Test odd values of K from 1 to 21 to avoid ties
k_values = range(1, 22, 2)
knn_accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    knn_accuracies.append(accuracy_score(y_test, knn.predict(X_test_scaled)))

# Plot K vs Accuracy
plt.figure(figsize=(8, 4))
plt.plot(k_values, knn_accuracies, marker='o', color='steelblue')
plt.title('KNN: Number of Neighbors (K) vs Accuracy')
plt.xlabel('K (Number of Neighbors)')
plt.ylabel('Test Accuracy')
plt.xticks(k_values)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(knn_accuracies)]
print(f"Best K: {best_k} with accuracy: {max(knn_accuracies):.4f}")

In [ ]:
# --- Train Final KNN Model with Best K ---
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train_scaled, y_train)
y_pred_knn = knn_model.predict(X_test_scaled)

print("=== KNN Classification Report ===")
print(classification_report(y_test, y_pred_knn, target_names=['No Purchase', 'Purchase']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_knn,
    display_labels=['No Purchase', 'Purchase'],
    cmap='Blues', ax=ax
)
ax.set_title(f'KNN Confusion Matrix (K={best_k})')
plt.tight_layout()
plt.show()

**Conclusion (KNN):** KNN classifies sessions by comparing them to the K most similar training sessions. The optimal K was identified by testing a range of values. KNN performs reasonably well on the majority class (No Purchase) but recall on the Purchase class may be limited due to class imbalance. It is a simple but interpretable baseline model.

### Model 2: Logistic Regression

In [ ]:
# --- Train Logistic Regression ---
# class_weight='balanced' compensates for the class imbalance
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, y_pred_lr, target_names=['No Purchase', 'Purchase']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_lr,
    display_labels=['No Purchase', 'Purchase'],
    cmap='Blues', ax=ax
)
ax.set_title('Logistic Regression Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# --- Feature Importance via Coefficients ---
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=coef_df.head(10), x='Coefficient', y='Feature', palette='coolwarm')
plt.title('Top 10 Features by Logistic Regression Coefficient (Absolute Value)')
plt.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

**Conclusion (Logistic Regression):** Logistic Regression models the probability of a purchase as a linear combination of features. By using `class_weight='balanced'`, the model is penalized more heavily for missing the minority (Purchase) class. The coefficient plot reveals which features most strongly push a session toward or away from a purchase — `PageValues` is expected to have the highest positive coefficient.

### Model 3: Decision Tree & Random Forest

In [ ]:
# --- Decision Tree ---
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt_model.fit(X_train, y_train)  # Trees don't require scaling
y_pred_dt = dt_model.predict(X_test)

print("=== Decision Tree Classification Report ===")
print(classification_report(y_test, y_pred_dt, target_names=['No Purchase', 'Purchase']))

# Confusion Matrix
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_dt,
    display_labels=['No Purchase', 'Purchase'],
    cmap='Blues', ax=ax
)
ax.set_title('Decision Tree Confusion Matrix (max_depth=5)')
plt.tight_layout()
plt.show()

In [ ]:
# --- Random Forest ---
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("=== Random Forest Classification Report ===")
print(classification_report(y_test, y_pred_rf, target_names=['No Purchase', 'Purchase']))

In [ ]:
# --- Random Forest Feature Importance ---
feat_imp_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(data=feat_imp_df.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Features by Random Forest Importance')
plt.tight_layout()
plt.show()

**Conclusion (Decision Tree & Random Forest):** The Decision Tree provides a simple, interpretable model — limiting depth to 5 prevents overfitting while still capturing key decision rules. Random Forest improves on this by averaging 100 trees, reducing variance significantly. The feature importance chart confirms `PageValues`, `BounceRates`, and `ExitRates` as the most informative predictors. Random Forest is expected to be the strongest classifier in this project.

### Model 4: K-Means Clustering (Unsupervised)

In [ ]:
# --- Elbow Method to Find Optimal K ---
inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_train_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(k_range, inertia_values, marker='o', color='steelblue')
plt.title('K-Means Elbow Method: Inertia vs Number of Clusters')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia (Within-Cluster Sum of Squares)')
plt.xticks(k_range)
plt.tight_layout()
plt.show()

In [ ]:
# --- Fit Final K-Means with K=3 ---
# 3 clusters represents a reasonable segmentation:
# likely: "browsers", "engaged non-buyers", and "high-intent buyers"
kmeans_model = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_model.fit(X_train_scaled)

# Assign cluster labels to the full dataset
df_model['Cluster'] = kmeans_model.fit_predict(scaler.transform(X))

# Analyze cluster composition
cluster_summary = df_model.groupby('Cluster').agg(
    Count=('Revenue', 'count'),
    Purchase_Rate=('Revenue', 'mean'),
    Avg_PageValues=('PageValues', 'mean'),
    Avg_BounceRate=('BounceRates', 'mean')
).round(3)

print("K-Means Cluster Summary:")
print(cluster_summary)

In [ ]:
# --- Visualize Clusters: PageValues vs BounceRates ---
fig, ax = plt.subplots(figsize=(9, 5))
scatter = ax.scatter(
    df_model['BounceRates'],
    df_model['PageValues'],
    c=df_model['Cluster'],
    cmap='Set1', alpha=0.3, s=10
)
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_title('K-Means Clusters: BounceRates vs PageValues')
ax.set_xlabel('Bounce Rate')
ax.set_ylabel('Page Values')
plt.tight_layout()
plt.show()

**Conclusion (K-Means Clustering):** K-Means reveals three natural visitor segments. The cluster with the highest `PageValues` and lowest `BounceRates` corresponds to high-intent shoppers with the highest purchase rate. The cluster with high bounce rates and low page values represents quick-exit visitors unlikely to convert. This segmentation is actionable — marketers can retarget the middle cluster (engaged but not yet converting) with promotions.

---
## Section 5 — Model Comparison

In [ ]:
# --- Compile Accuracy Scores ---
model_names  = ['KNN', 'Logistic Regression', 'Decision Tree', 'Random Forest']
y_preds      = [y_pred_knn, y_pred_lr, y_pred_dt, y_pred_rf]

results = []
for name, y_pred in zip(model_names, y_preds):
    report = classification_report(y_test, y_pred, output_dict=True)
    results.append({
        'Model':     name,
        'Accuracy':  round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(report['1']['precision'], 4),
        'Recall':    round(report['1']['recall'], 4),
        'F1-Score':  round(report['1']['f1-score'], 4)
    })

results_df = pd.DataFrame(results).set_index('Model')
print("=== Model Comparison (Purchase Class) ===")
print(results_df)

In [ ]:
# --- Bar Chart: F1-Score Comparison ---
fig, ax = plt.subplots(figsize=(9, 5))
results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score']].plot(
    kind='bar', ax=ax, edgecolor='black'
)
ax.set_title('Model Performance Comparison (Purchase Class)')
ax.set_ylabel('Score')
ax.set_ylim(0, 1.05)
ax.set_xticklabels(model_names, rotation=15)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

**Conclusion (Model Comparison):** Random Forest achieves the best overall balance of precision, recall, and F1-score on the Purchase class, making it the recommended model for deployment. Logistic Regression with balanced class weights tends to achieve higher recall (catches more true purchases) at the cost of some precision. KNN performs reasonably but is sensitive to the curse of dimensionality with 17 features. Decision Tree is the most interpretable but slightly underperforms the ensemble method.

---
## Section 6 — MongoDB Integration

In [ ]:
# --- MongoDB: Store Dataset and Query Results ---
# Requires: pip install pymongo
# Assumes a local MongoDB instance running on default port 27017
# To start MongoDB locally: mongod --dbpath /data/db

from pymongo import MongoClient
import json

# Connect to local MongoDB
client = MongoClient('mongodb://localhost:27017/')
db = client['ecommerce_db']
collection = db['shoppers']

print("Connected to MongoDB.")
print(f"Database: ecommerce_db | Collection: shoppers")

In [ ]:
# --- Insert Dataset into MongoDB ---
# Convert DataFrame to list of dicts (documents)
df_mongo = df.copy()
df_mongo['Revenue'] = df_mongo['Revenue'].astype(bool)  # Keep as bool for natural document feel
records = df_mongo.to_dict(orient='records')

# Clear collection before inserting to avoid duplicates
collection.drop()
result = collection.insert_many(records)

print(f"Inserted {len(result.inserted_ids)} documents into MongoDB.")

In [ ]:
# --- Query 1: Count sessions that resulted in a purchase ---
purchase_count = collection.count_documents({'Revenue': True})
no_purchase_count = collection.count_documents({'Revenue': False})
print(f"Sessions with purchase:    {purchase_count}")
print(f"Sessions without purchase: {no_purchase_count}")

In [ ]:
# --- Query 2: Average PageValues for buyers vs non-buyers ---
pipeline = [
    {
        '$group': {
            '_id': '$Revenue',
            'avg_page_values': {'$avg': '$PageValues'},
            'avg_bounce_rate': {'$avg': '$BounceRates'},
            'session_count':   {'$sum': 1}
        }
    },
    {'$sort': {'_id': 1}}
]

results_mongo = list(collection.aggregate(pipeline))
for doc in results_mongo:
    label = 'Purchase' if doc['_id'] else 'No Purchase'
    print(f"\n{label}:")
    print(f"  Sessions:          {doc['session_count']}")
    print(f"  Avg PageValues:    {doc['avg_page_values']:.3f}")
    print(f"  Avg Bounce Rate:   {doc['avg_bounce_rate']:.4f}")

In [ ]:
# --- Query 3: Sessions from returning visitors in November with purchase ---
nov_returning = collection.count_documents({
    'Month': 'Nov',
    'VisitorType': 'Returning_Visitor',
    'Revenue': True
})
print(f"Returning visitors in November who made a purchase: {nov_returning}")

In [ ]:
# --- Query 4: Top 5 sessions by PageValues that resulted in a purchase ---
top_sessions = list(
    collection.find(
        {'Revenue': True},
        {'_id': 0, 'Month': 1, 'VisitorType': 1, 'PageValues': 1, 'BounceRates': 1}
    ).sort('PageValues', -1).limit(5)
)

print("Top 5 Purchase Sessions by PageValues:")
for i, doc in enumerate(top_sessions, 1):
    print(f"  {i}. {doc}")

**Conclusion (MongoDB):** The dataset was successfully loaded into MongoDB as a collection of session documents. Four queries were executed: counting purchases, aggregating average metrics by outcome, filtering by visitor segment and month, and ranking high-value sessions. MongoDB's document model is well-suited to this data since each session is a natural JSON document, and the aggregation pipeline efficiently computes group statistics without the need for joins.

---
## Section 7 — Conclusions

### Summary of Findings

1. **Class Imbalance:** Only ~16% of sessions result in a purchase. Standard accuracy alone is misleading — F1-score on the purchase class is the more meaningful metric.

2. **Key Predictors:** `PageValues`, `BounceRates`, and `ExitRates` are consistently the most important features across all models. `PageValues` alone is a strong signal of purchase intent.

3. **Seasonal Patterns:** November and December show the highest conversion rates, aligning with holiday shopping seasons. Marketing budgets should be weighted toward these months.

4. **Best Model:** Random Forest achieved the best balance of precision and recall on the Purchase class. It is robust to the class imbalance and handles the non-linear relationships in the data well.

5. **Visitor Segmentation:** K-Means identified three meaningful visitor clusters — casual browsers, engaged non-buyers, and high-intent shoppers — which can directly inform retargeting strategies.

6. **Business Recommendation:** Retailers should focus on sessions with high `PageValues` and low `BounceRates` as the highest-priority targets for real-time promotions or personalized recommendations.

---
### References

**Academic:**
- Sakar, C.O., Polat, S.O., Katircioglu, M., & Kastro, Y. (2019). Real-time prediction of online shoppers' purchasing intention using multilayer perceptron and LSTM recurrent neural networks. *Neural Computing and Applications*, 31, 6893–6908. https://doi.org/10.1007/s00521-018-3523-0

**Dataset:**
- UCI Machine Learning Repository. (2018). *Online Shoppers Purchasing Intention Dataset*. https://archive.ics.uci.edu/dataset/468/online+shoppers+purchasing+intention+dataset

**Online:**
- scikit-learn documentation: https://scikit-learn.org/stable/
- MongoDB PyMongo documentation: https://pymongo.readthedocs.io/en/stable/
- Seaborn documentation: https://seaborn.pydata.org/

---
### Credits

> *Dataset sourced from the UCI Machine Learning Repository. All code was written independently for CISD 43 – Big Data (Spring, 2026) at California Polytechnic State University, Pomona.*

In [ ]:
# End of Project